In [ ]:
# Res-UNet 3D with Multimodal Fusion - Complete Pipeline
# Optimized for Kaggle T4 GPU (~8h runtime)

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os
import math
import time
import pathlib
from glob import glob
from collections import defaultdict
from typing import Sequence, Tuple, Dict, List, Optional
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from torch.optim.lr_scheduler import CosineAnnealingLR
from scipy.ndimage import binary_fill_holes, label as cc_label

# Medical image readers
try:
    import nibabel as nib  # NIfTI
except Exception:
    nib = None
try:
    import SimpleITK as sitk  # MHA/MHD, NIfTI
except Exception:
    sitk = None

# Device selection (CUDA for Kaggle T4)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")


In [ ]:
# Model: ResUNet3D

class ConvNormAct3d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, stride: int = 1,
                 padding: int | None = None, bias: bool = False, norm: str = "bn", act: str = "relu") -> None:
        super().__init__()
        if padding is None:
            padding = kernel_size // 2
        self.conv = nn.Conv3d(in_channels, out_channels, kernel_size=kernel_size, stride=stride,
                              padding=padding, bias=bias)
        if norm == "bn":
            self.norm = nn.BatchNorm3d(out_channels)
        elif norm == "in":
            self.norm = nn.InstanceNorm3d(out_channels, affine=True)
        else:
            raise ValueError(f"Unsupported norm: {norm}")
        if act == "relu":
            self.act = nn.ReLU(inplace=True)
        elif act == "lrelu":
            self.act = nn.LeakyReLU(0.01, inplace=True)
        elif act == "elu":
            self.act = nn.ELU(inplace=True)
        else:
            raise ValueError(f"Unsupported act: {act}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)
        x = self.norm(x)
        x = self.act(x)
        return x


class ResidualBlock3d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, norm: str = "bn", act: str = "relu",
                 dropout: float | None = None) -> None:
        super().__init__()
        self.conv1 = ConvNormAct3d(in_channels, out_channels, kernel_size=3, stride=1, norm=norm, act=act)
        self.conv2 = nn.Conv3d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        if norm == "bn":
            self.norm2 = nn.BatchNorm3d(out_channels)
        else:
            self.norm2 = nn.InstanceNorm3d(out_channels, affine=True)
        self.act = nn.ReLU(inplace=True) if act == "relu" else (nn.LeakyReLU(0.01, inplace=True) if act == "lrelu" else nn.ELU(inplace=True))
        self.dropout = nn.Dropout3d(p=dropout) if dropout and dropout > 0 else None

        if in_channels != out_channels:
            self.proj = nn.Conv3d(in_channels, out_channels, kernel_size=1, stride=1, bias=False)
        else:
            self.proj = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = self.conv1(x)
        if self.dropout is not None:
            out = self.dropout(out)
        out = self.conv2(out)
        out = self.norm2(out)
        if self.proj is not None:
            identity = self.proj(identity)
        out = out + identity
        out = self.act(out)
        return out


class DownBlock3d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, norm: str = "bn", act: str = "relu",
                 dropout: float | None = None, down_stride: int = 2) -> None:
        super().__init__()
        self.down = nn.Conv3d(in_channels, in_channels, kernel_size=3, stride=down_stride, padding=1, bias=False)
        self.block = ResidualBlock3d(in_channels, out_channels, norm=norm, act=act, dropout=dropout)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        skip = x
        x = self.down(x)
        x = self.block(x)
        return x, skip


class UpBlock3d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, skip_channels: int, norm: str = "bn", act: str = "relu",
                 dropout: float | None = None) -> None:
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, kernel_size=2, stride=2)
        self.block = ResidualBlock3d(out_channels + skip_channels, out_channels, norm=norm, act=act, dropout=dropout)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)
        dz = skip.shape[-3] - x.shape[-3]
        dy = skip.shape[-2] - x.shape[-2]
        dx = skip.shape[-1] - x.shape[-1]
        if dz != 0 or dy != 0 or dx != 0:
            x = F.pad(x, (0, dx, 0, dy, 0, dz))
        x = torch.cat([x, skip], dim=1)
        x = self.block(x)
        return x


class ResUNet3D(nn.Module):
    def __init__(self, in_channels: int = 1, num_classes: int = 4, base_filters: int = 32,
                 norm: str = "bn", act: str = "relu", dropout_at: int | None = 256, dropout_p: float = 0.1) -> None:
        super().__init__()
        f1, f2, f3, f4, f5 = base_filters, base_filters * 2, base_filters * 4, base_filters * 8, base_filters * 16
        do = lambda c: (dropout_p if (dropout_at is not None and c >= dropout_at) else None)

        self.stem = ResidualBlock3d(in_channels, f1, norm=norm, act=act, dropout=do(f1))

        self.down1 = DownBlock3d(f1, f2, norm=norm, act=act, dropout=do(f2))
        self.down2 = DownBlock3d(f2, f3, norm=norm, act=act, dropout=do(f3))
        self.down3 = DownBlock3d(f3, f4, norm=norm, act=act, dropout=do(f4))
        self.down4 = DownBlock3d(f4, f5, norm=norm, act=act, dropout=do(f5))

        self.up1 = UpBlock3d(f5, f4, skip_channels=f4, norm=norm, act=act, dropout=do(f4))
        self.up2 = UpBlock3d(f4, f3, skip_channels=f3, norm=norm, act=act, dropout=do(f3))
        self.up3 = UpBlock3d(f3, f2, skip_channels=f2, norm=norm, act=act, dropout=do(f2))
        self.up4 = UpBlock3d(f2, f1, skip_channels=f1, norm=norm, act=act, dropout=do(f1))

        self.head = nn.Conv3d(f1, num_classes, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x0 = self.stem(x)

        x1, s0 = self.down1(x0)
        x2, s1 = self.down2(x1)
        x3, s2 = self.down3(x2)
        x4, s3 = self.down4(x3)

        y3 = self.up1(x4, s3)
        y2 = self.up2(y3, s2)
        y1 = self.up3(y2, s1)
        y0 = self.up4(y1, s0)

        logits = self.head(y0)
        return logits


def build_resunet3d(in_channels: int = 1, num_classes: int = 4, base_filters: int = 32,
                    norm: str = "bn", act: str = "relu", dropout_at: int | None = 256, dropout_p: float = 0.1) -> ResUNet3D:
    return ResUNet3D(in_channels=in_channels, num_classes=num_classes, base_filters=base_filters,
                     norm=norm, act=act, dropout_at=dropout_at, dropout_p=dropout_p)



In [ ]:
# Losses: Dice, Focal, Combined

class DiceLoss(nn.Module):
    def __init__(self, num_classes: int, smooth: float = 1.0, reduction: str = "mean") -> None:
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        if targets.dim() == logits.dim():
            target_one_hot = targets
        else:
            target_one_hot = F.one_hot(targets.long(), num_classes=self.num_classes).permute(0, 4, 1, 2, 3).float()

        probs = torch.softmax(logits, dim=1)
        dims = (0, 2, 3, 4)
        intersection = torch.sum(probs * target_one_hot, dim=dims)
        cardinality = torch.sum(probs + target_one_hot, dim=dims)
        dice_per_class = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        loss_per_class = 1.0 - dice_per_class

        if self.reduction == "mean":
            return loss_per_class.mean()
        if self.reduction == "sum":
            return loss_per_class.sum()
        return loss_per_class


class FocalLoss(nn.Module):
    def __init__(self, num_classes: int, alpha: torch.Tensor | None = None, gamma: float = 2.0,
                 reduction: str = "mean") -> None:
        super().__init__()
        self.num_classes = num_classes
        if alpha is None:
            self.alpha = torch.ones(num_classes)
        else:
            self.alpha = alpha.view(-1)
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        log_probs = F.log_softmax(logits, dim=1)
        probs = log_probs.exp()

        if targets.dim() == logits.dim():
            targets_indices = torch.argmax(targets, dim=1)
        else:
            targets_indices = targets.long()

        log_pt = log_probs.gather(1, targets_indices.unsqueeze(1)).squeeze(1)
        pt = probs.gather(1, targets_indices.unsqueeze(1)).squeeze(1)

        alpha = self.alpha.to(logits.device)
        at = alpha.gather(0, targets_indices.view(-1)).view_as(pt)

        focal_term = (1 - pt) ** self.gamma
        loss = -at * focal_term * log_pt

        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss


class CombinedDiceFocalLoss(nn.Module):
    def __init__(self, num_classes: int, dice_weight: float = 0.5, focal_weight: float = 0.5,
                 focal_alpha: torch.Tensor | None = None, focal_gamma: float = 2.0) -> None:
        super().__init__()
        assert abs(dice_weight + focal_weight - 1.0) < 1e-6
        self.dice = DiceLoss(num_classes=num_classes)
        self.focal = FocalLoss(num_classes=num_classes, alpha=focal_alpha, gamma=focal_gamma)
        self.dw = dice_weight
        self.fw = focal_weight

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        return self.dw * self.dice(logits, targets) + self.fw * self.focal(logits, targets)



In [ ]:
# Fusion utilities (static, Dice-weighted)
from typing import Sequence, Tuple

@torch.no_grad()
def normalize_weights(weights: Sequence[float]) -> torch.Tensor:
    w = torch.tensor(weights, dtype=torch.float32)
    w = torch.clamp(w, min=0.0)
    s = w.sum()
    if s <= 0:
        # fallback to uniform
        w = torch.ones_like(w) / len(w)
    else:
        w = w / s
    return w

@torch.no_grad()
def fusion_weights_from_dice(dice_per_modality: Sequence[float]) -> torch.Tensor:
    # proportional to Dice, normalized to 1
    return normalize_weights(dice_per_modality)

@torch.no_grad()
def fuse_modalities_linear(t1: torch.Tensor, t2: torch.Tensor, t1ce: torch.Tensor, flair: torch.Tensor,
                           weights: Sequence[float] | torch.Tensor) -> torch.Tensor:
    """
    Inputs are tensors with identical shapes (D,H,W) or (1,D,H,W) or (B,1,D,H,W).
    Returns fused tensor with same batch/shape, channel=1.
    """
    if not torch.is_tensor(weights):
        w = normalize_weights(weights)
    else:
        w = normalize_weights(weights.tolist())
    vols = [t1, t2, t1ce, flair]
    # ensure shape (B,1,D,H,W)
    proc = []
    for v in vols:
        if v.dim() == 3:
            v = v.unsqueeze(0).unsqueeze(0)
        elif v.dim() == 4:
            v = v.unsqueeze(1)
        proc.append(v)
    fused = w[0] * proc[0] + w[1] * proc[1] + w[2] * proc[2] + w[3] * proc[3]
    return fused



In [ ]:
# Dataset/Loader with preprocessing hooks and tumor-centric patching

class PreprocessConfig:
    def __init__(self,
                 target_spacing=(1.0, 1.0, 1.0),
                 clip_percentiles=(0.5, 99.5),
                 zscore=True,
                 apply_n4=True,
                 skull_strip=True):
        self.target_spacing = target_spacing
        self.clip_percentiles = clip_percentiles
        self.zscore = zscore
        self.apply_n4 = apply_n4
        self.skull_strip = skull_strip


def zscore_normalize(vol: np.ndarray, mask: Optional[np.ndarray] = None) -> np.ndarray:
    if mask is not None:
        vox = vol[mask > 0]
    else:
        vox = vol
    mean = float(vox.mean()) if vox.size > 0 else 0.0
    std = float(vox.std()) if vox.size > 0 else 1.0
    std = std if std > 1e-6 else 1.0
    out = (vol - mean) / std
    return out


def percentile_clip(vol: np.ndarray, low: float, high: float) -> np.ndarray:
    lo, hi = np.percentile(vol, [low, high])
    vol = np.clip(vol, lo, hi)
    return vol


class TumorPatchSampler:
    def __init__(self, patch_size=(128, 128, 128), tumor_fraction: float = 0.7):  # Kaggle T4: larger patches for quality within 8h
        self.patch_size = patch_size
        self.tumor_fraction = tumor_fraction

    def sample_indices(self, label: np.ndarray, num_patches: int) -> List[tuple]:
        D, H, W = label.shape
        pd, ph, pw = self.patch_size
        indices: List[tuple] = []
        tumor_coords = np.argwhere(label > 0)
        for i in range(num_patches):
            if np.random.rand() < self.tumor_fraction and tumor_coords.size > 0:
                z, y, x = tumor_coords[np.random.randint(0, tumor_coords.shape[0])]
                z0 = max(0, z - pd // 2)
                y0 = max(0, y - ph // 2)
                x0 = max(0, x - pw // 2)
            else:
                z0 = np.random.randint(0, max(1, D - pd + 1))
                y0 = np.random.randint(0, max(1, H - ph + 1))
                x0 = np.random.randint(0, max(1, W - pw + 1))
            indices.append((z0, y0, x0))
        return indices

    def crop(self, vol: np.ndarray, z0: int, y0: int, x0: int) -> np.ndarray:
        pd, ph, pw = self.patch_size
        return vol[z0:z0+pd, y0:y0+ph, x0:x0+pw]



In [ ]:
# Training/Evaluation scaffold (5-fold CV-ready, AMP + CUDA)

@dataclass
class TrainConfig:
    lr: float = 1e-4
    weight_decay: float = 1e-5
    epochs: int = 80  # Kaggle T4 target; adjust with folds to fit ~8h
    batch_size: int = 1  # Safer default for 3D patches on Kaggle T4
    grad_clip: float = 1.0
    num_workers: int = 0  # Single-process loading to reduce RAM pressure on Kaggle
    amp: bool = True
    checkpoint_every: int = 25  # save every N epochs on top of best/last


def configure_optimizer(model: nn.Module, cfg: TrainConfig):
    return torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)


def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer, loss_fn, device, cfg: TrainConfig):
    model.train()
    use_amp = cfg.amp
    total_loss = 0.0
    if device.type == "cuda" and use_amp:
        scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    else:
        scaler = None
    for batch in loader:
        imgs = batch["image"].unsqueeze(1).to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if device.type == "cuda" and use_amp:
            with torch.autocast(device_type='cuda', enabled=True):
                logits = model(imgs)
                loss = loss_fn(logits, labels)
            scaler.scale(loss).backward()
            if cfg.grad_clip is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
        elif device.type == "mps" and use_amp:
            # AMP on MPS uses torch.autocast, no GradScaler
            with torch.autocast(device_type="mps", enabled=True):
                logits = model(imgs)
                loss = loss_fn(logits, labels)
            loss.backward()
            if cfg.grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()
        else:
            logits = model(imgs)
            loss = loss_fn(logits, labels)
            loss.backward()
            if cfg.grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()
        total_loss += float(loss.detach().cpu())
    return total_loss / max(1, len(loader))


def validate_one_epoch(model: nn.Module, loader: DataLoader, loss_fn, device, amp: bool):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            imgs = batch["image"].unsqueeze(1).to(device, non_blocking=True)
            labels = batch["label"].to(device, non_blocking=True)
            if device.type == "cuda" and amp:
                with torch.autocast(device_type='cuda', enabled=True):
                    logits = model(imgs)
                    loss = loss_fn(logits, labels)
            elif device.type == "mps" and amp:
                with torch.autocast(device_type="mps", enabled=True):
                    logits = model(imgs)
                    loss = loss_fn(logits, labels)
            else:
                logits = model(imgs)
                loss = loss_fn(logits, labels)
            total_loss += float(loss.detach().cpu())
    return total_loss / max(1, len(loader))



In [ ]:
# Full training loop with scheduler, early stopping, checkpoints

class EarlyStopping:
    def __init__(self, patience: int = 20, mode: str = "max"):
        self.patience = patience
        self.mode = mode
        self.best = -math.inf if mode == "max" else math.inf
        self.count = 0

    def step(self, value: float) -> bool:
        improved = (value > self.best) if self.mode == "max" else (value < self.best)
        if improved:
            self.best = value
            self.count = 0
            return False
        self.count += 1
        return self.count > self.patience


def save_checkpoint(path: str, model: nn.Module, optimizer, epoch: int, best_metric: float):
    path = str(path)
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "best_metric": best_metric,
    }, path)


def maybe_checkpoint_every(epoch: int, cfg: TrainConfig, out_dir: pathlib.Path, model: nn.Module, optimizer, best_metric: float):
    if cfg.checkpoint_every and epoch % cfg.checkpoint_every == 0:
        save_checkpoint(out_dir / f"epoch_{epoch:04d}.pt", model, optimizer, epoch, best_metric)


# Example wire-up (expects a Dataset implementation)
def run_training(train_ds: Dataset, val_ds: Dataset, num_classes=4, in_channels=1, out_dir="./checkpoints", cfg: TrainConfig = TrainConfig()):
    out = pathlib.Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        persistent_workers=False,
        pin_memory=False,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=max(1, cfg.batch_size // 2),
        shuffle=False,
        num_workers=cfg.num_workers,
        persistent_workers=False,
        pin_memory=False,
    )

    model = build_resunet3d(in_channels=in_channels, num_classes=num_classes, base_filters=16).to(device)
    optimizer = configure_optimizer(model, cfg)
    scheduler = CosineAnnealingLR(optimizer, T_max=max(1, cfg.epochs), eta_min=1e-6)
    loss_fn = CombinedDiceFocalLoss(num_classes=num_classes, dice_weight=0.5, focal_weight=0.5)

    stopper = EarlyStopping(patience=10, mode="min")  # patience reduced to suit 30-epoch runs
    best_val = math.inf

    for epoch in range(1, cfg.epochs + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device, cfg)
        val_loss = validate_one_epoch(model, val_loader, loss_fn, device, amp=cfg.amp)
        scheduler.step()
        is_best = val_loss < best_val
        if is_best:
            best_val = val_loss
            save_checkpoint(out / "best.pt", model, optimizer, epoch, best_val)
        save_checkpoint(out / "last.pt", model, optimizer, epoch, best_val)
        maybe_checkpoint_every(epoch, cfg, out, model, optimizer, best_val)
        dt = time.time() - t0
        print(f"Epoch {epoch}/{cfg.epochs} | train {train_loss:.4f} | val {val_loss:.4f} | lr {optimizer.param_groups[0]['lr']:.2e} | {dt:.1f}s")
        if stopper.step(val_loss):
            print("Early stopping triggered.")
            break
    print(f"Best val loss: {best_val:.4f}")



In [ ]:
# BraTS dataset IO and validation

MODALITY_SUFFIXES = {
    # NIfTI style
    "t1": ("_t1", "t1"),
    "t2": ("_t2", "t2"),
    "t1ce": ("_t1ce", "t1ce"),
    "flair": ("_flair", "flair"),
    "seg": ("_seg", "seg"),
    # BraTS2015 MHA style (OT is segmentation)
    "t1_mha": ("_T1", "T1"),
    "t2_mha": ("_T2", "T2"),
    "t1ce_mha": ("_T1c", "T1c"),
    "flair_mha": ("_Flair", "Flair"),
    "seg_mha": ("_OT", "OT"),
}


def find_cases(root: str) -> list[dict]:
    """Find and group BraTS 2015 .mha files per case (T1, T1c, T2, FLAIR, Seg)."""
    files = glob(os.path.join(root, "**", "*.mha"), recursive=True)
    by_case = defaultdict(dict)

    for fp in files:
        fname = os.path.basename(fp)
        lname = fname.lower()

        # Identify modality based on BraTS 2015 naming pattern
        if "mr_t1c" in lname:
            modality = "t1ce"
        elif "mr_t1" in lname and not "t1c" in lname:
            modality = "t1"
        elif "mr_t2" in lname:
            modality = "t2"
        elif "mr_flair" in lname:
            modality = "flair"
        elif ".ot." in lname or "ot." in lname:
            modality = "seg"
        else:
            continue

        # Case ID = the folder containing the files
        case_id = os.path.basename(os.path.dirname(fp))
        by_case[case_id][modality] = fp

    # Keep only complete cases
    cases = [v for v in by_case.values() if all(m in v for m in ("t1", "t2", "t1ce", "flair", "seg"))]
    return cases


def validate_braTS_root(root: str) -> None:
    cases = find_cases(root)
    print(f"Found {len(cases)} complete cases in {root}")
    if len(cases) == 0:
        raise RuntimeError("No complete cases (t1,t2,t1ce,flair,seg) found.")



In [ ]:
# Preprocessing stubs: resample, N4, skull-strip (placeholders for integration)
# NOTE: For full functionality, integrate with SimpleITK/ANTsPy/HD-BET as available on your system.

def resample_to_spacing_nii(img_nii, target_spacing=(1.0,1.0,1.0)):
    # Placeholder: return as-is. Replace with SimpleITK resample if available.
    return img_nii


def apply_n4_bias_correction_nii(img_nii):
    # Placeholder: return as-is. Replace with SimpleITK N4BiasFieldCorrection if available.
    return img_nii


def skull_strip_with_mask_nii(img_nii, brain_mask=None):
    # Placeholder: if mask provided, apply; otherwise return as-is.
    return img_nii



In [ ]:
# K-fold split builder


def build_kfold_splits(cases: list[dict], k: int = 5, seed: int = 42):
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)
    cases_idx = list(range(len(cases)))
    folds = []
    for tr_idx, va_idx in kf.split(cases_idx):
        folds.append({
            "train": [cases[i] for i in tr_idx],
            "val": [cases[i] for i in va_idx],
        })
    return folds



In [ ]:
# Metrics (Dice per class), HD95 (placeholder), and postprocessing


def dice_per_class(pred: torch.Tensor, target: torch.Tensor, num_classes: int) -> torch.Tensor:
    # pred: logits (N,C,D,H,W) or probs; we argmax
    if pred.shape[1] == num_classes:
        pred_lbl = torch.argmax(pred, dim=1)
    else:
        pred_lbl = pred.long()
    target_lbl = target.long()
    dices = []
    for c in range(num_classes):
        p = (pred_lbl == c)
        t = (target_lbl == c)
        inter = (p & t).sum().float()
        denom = p.sum().float() + t.sum().float()
        d = (2 * inter + 1.0) / (denom + 1.0)
        dices.append(d)
    return torch.stack(dices)


def hd95_placeholder(pred: torch.Tensor, target: torch.Tensor, voxel_spacing=(1.0,1.0,1.0), num_classes: int = 4):
    # Placeholder. Proper HD95 requires surface distance (e.g., medpy/surface-distance or SimpleITK filters)
    return [float('nan')] * num_classes


def keep_largest_component_per_class(seg: np.ndarray, num_classes: int) -> np.ndarray:
    out = np.zeros_like(seg)
    for c in range(num_classes):
        mask = (seg == c).astype(np.uint8)
        if mask.sum() == 0:
            continue
        labeled, n = cc_label(mask)
        if n <= 1:
            out[mask > 0] = c
            continue
        sizes = [(labeled == i).sum() for i in range(1, n+1)]
        largest_idx = 1 + int(np.argmax(sizes))
        out[labeled == largest_idx] = c
    return out


def fill_holes_per_class(seg: np.ndarray, num_classes: int) -> np.ndarray:
    out = np.zeros_like(seg)
    for c in range(num_classes):
        mask = (seg == c)
        out[binary_fill_holes(mask)] = c
    return out



In [ ]:
# Padding utility to enforce fixed patch size
import numpy as _np

def pad_to_min_shape(vol: _np.ndarray, min_shape: tuple[int, int, int], constant: float = 0.0) -> _np.ndarray:
    d, h, w = vol.shape
    md, mh, mw = min_shape
    pd = max(0, md - d)
    ph = max(0, mh - h)
    pw = max(0, mw - w)
    if pd == 0 and ph == 0 and pw == 0:
        return vol
    # pad at the end to reach at least the min size
    return _np.pad(vol, ((0, pd), (0, ph), (0, pw)), mode='constant', constant_values=constant)


In [ ]:
# Sliding-window inference for full volumes (simple version)

def sliding_window_inference(volume: torch.Tensor, model: nn.Module, window=(128,128,128), overlap=0.5, num_classes=4):
    """
    volume: (1, D, H, W) torch tensor. Returns logits (C,D,H,W) on CPU.
    """
    model.eval()
    with torch.no_grad():
        _, D, H, W = volume.shape
        pd, ph, pw = window
        sd = max(1, int(pd * (1 - overlap)))
        sh = max(1, int(ph * (1 - overlap)))
        sw = max(1, int(pw * (1 - overlap)))
        out_logits = torch.zeros((num_classes, D, H, W), device=device)
        out_norm = torch.zeros((1, D, H, W), device=device)
        for z0 in range(0, max(1, D - pd + 1), sd):
            for y0 in range(0, max(1, H - ph + 1), sh):
                for x0 in range(0, max(1, W - pw + 1), sw):
                    patch = volume[:, z0:z0+pd, y0:y0+ph, x0:x0+pw].unsqueeze(0).to(device)
                    logits = model(patch)
                    out_logits[:, z0:z0+pd, y0:y0+ph, x0:x0+pw] += logits.squeeze(0)
                    out_norm[:, z0:z0+pd, y0:y0+ph, x0:x0+pw] += 1
        out_norm[out_norm == 0] = 1
        out_logits = out_logits / out_norm
        return out_logits.cpu()



In [ ]:
# Fusion weight computation from per-modality Dice scores

def compute_fusion_weights_from_val_dice(dice_t1: float, dice_t2: float, dice_t1ce: float, dice_flair: float):
    weights = fusion_weights_from_dice([dice_t1, dice_t2, dice_t1ce, dice_flair])
    print("Fusion weights (t1, t2, t1ce, flair):", weights.tolist())
    return weights



In [ ]:
# BraTS Dataset class with on-the-fly fusion and tumor-centric patching
class BraTSDataset(Dataset):
    def __init__(self, cases: list[dict], weights: torch.Tensor, sampler: TumorPatchSampler,
                 preprocess: PreprocessConfig = PreprocessConfig(), num_patches_per_volume: int = 16,
                 training: bool = True):
        assert len(weights) == 4
        self.cases = cases
        self.weights = weights.float()
        self.sampler = sampler
        self.pre = preprocess
        self.num_patches_per_volume = num_patches_per_volume
        self.training = training

    def __len__(self):
        return len(self.cases) * (self.num_patches_per_volume if self.training else 1)

    def _load_case(self, case: dict):
        # Support NIfTI via nibabel and MHA/MHD via SimpleITK
        if case['t1'].lower().endswith(('.mha', '.mhd')):
            if sitk is None:
                raise RuntimeError("SimpleITK is required to read MHA/MHD files. Please install SimpleITK.")
            def read_sitk(fp):
                return sitk.ReadImage(fp)
            t1 = read_sitk(case['t1'])
            t2 = read_sitk(case['t2'])
            t1ce = read_sitk(case['t1ce'])
            flair = read_sitk(case['flair'])
            seg = read_sitk(case['seg'])
            # Preprocessing stubs (no-ops by default)
            t1 = resample_to_spacing_nii(t1, self.pre.target_spacing)
            t2 = resample_to_spacing_nii(t2, self.pre.target_spacing)
            t1ce = resample_to_spacing_nii(t1ce, self.pre.target_spacing)
            flair = resample_to_spacing_nii(flair, self.pre.target_spacing)
            seg = resample_to_spacing_nii(seg, self.pre.target_spacing)
            # Get numpy arrays
            t1v = sitk.GetArrayFromImage(t1).astype(np.float32)  # (D,H,W) in array order
            t2v = sitk.GetArrayFromImage(t2).astype(np.float32)
            t1cev = sitk.GetArrayFromImage(t1ce).astype(np.float32)
            flairv = sitk.GetArrayFromImage(flair).astype(np.float32)
            segv = sitk.GetArrayFromImage(seg).astype(np.int16)
        else:
            if nib is None:
                raise RuntimeError("nibabel is required to read NIfTI files. Please install nibabel.")
            t1 = nib.load(case['t1'])
            t2 = nib.load(case['t2'])
            t1ce = nib.load(case['t1ce'])
            flair = nib.load(case['flair'])
            seg = nib.load(case['seg'])
            # Preprocessing stubs (no-ops by default)
            t1 = resample_to_spacing_nii(t1, self.pre.target_spacing)
            t2 = resample_to_spacing_nii(t2, self.pre.target_spacing)
            t1ce = resample_to_spacing_nii(t1ce, self.pre.target_spacing)
            flair = resample_to_spacing_nii(flair, self.pre.target_spacing)
            seg = resample_to_spacing_nii(seg, self.pre.target_spacing)
            # Get numpy arrays
            t1v = t1.get_fdata().astype(np.float32)
            t2v = t2.get_fdata().astype(np.float32)
            t1cev = t1ce.get_fdata().astype(np.float32)
            flairv = flair.get_fdata().astype(np.float32)
            segv = seg.get_fdata().astype(np.int16)
        
        # Pad volumes to ensure minimum patch size
        min_shape = self.sampler.patch_size
        t1v = pad_to_min_shape(t1v, min_shape, constant=0.0)
        t2v = pad_to_min_shape(t2v, min_shape, constant=0.0)
        t1cev = pad_to_min_shape(t1cev, min_shape, constant=0.0)
        flairv = pad_to_min_shape(flairv, min_shape, constant=0)
        segv = pad_to_min_shape(segv, min_shape, constant=0)
        
        # Clip and zscore
        if self.pre.clip_percentiles is not None:
            lo, hi = self.pre.clip_percentiles
            t1v = percentile_clip(t1v, lo, hi)
            t2v = percentile_clip(t2v, lo, hi)
            t1cev = percentile_clip(t1cev, lo, hi)
            flairv = percentile_clip(flairv, lo, hi)
        if self.pre.zscore:
            brain_mask = (t1v > 0) | (t2v > 0) | (t1cev > 0) | (flairv > 0)
            t1v = zscore_normalize(t1v, brain_mask)
            t2v = zscore_normalize(t2v, brain_mask)
            t1cev = zscore_normalize(t1cev, brain_mask)
            flairv = zscore_normalize(flairv, brain_mask)
        return t1v, t2v, t1cev, flairv, segv

    def __getitem__(self, idx):
        case_idx = idx // (self.num_patches_per_volume if self.training else 1)
        case = self.cases[case_idx]
        t1v, t2v, t1cev, flairv, segv = self._load_case(case)
        # Sample patch index
        if self.training:
            z0, y0, x0 = self.sampler.sample_indices(segv, 1)[0]
            img_patch = None  # fuse after cropping for efficiency
            pd, ph, pw = self.sampler.patch_size
            t1p = t1v[z0:z0+pd, y0:y0+ph, x0:x0+pw]
            t2p = t2v[z0:z0+pd, y0:y0+ph, x0:x0+pw]
            t1cep = t1cev[z0:z0+pd, y0:y0+ph, x0:x0+pw]
            flairp = flairv[z0:z0+pd, y0:y0+ph, x0:x0+pw]
            segp = segv[z0:z0+pd, y0:y0+ph, x0:x0+pw]
            # enforce exact patch size by padding if crop hits boundary
            min_shape = (pd, ph, pw)
            t1p = pad_to_min_shape(t1p, min_shape, constant=0.0)
            t2p = pad_to_min_shape(t2p, min_shape, constant=0.0)
            t1cep = pad_to_min_shape(t1cep, min_shape, constant=0.0)
            flairp = pad_to_min_shape(flairp, min_shape, constant=0.0)
            segp = pad_to_min_shape(segp, min_shape, constant=0)
        else:
            # full volume (already padded in _load_case)
            t1p, t2p, t1cep, flairp, segp = t1v, t2v, t1cev, flairv, segv
        # Fuse
        t1t = torch.from_numpy(t1p).float()
        t2t = torch.from_numpy(t2p).float()
        t1cet = torch.from_numpy(t1cep).float()
        flairt = torch.from_numpy(flairp).float()
        fused = fuse_modalities_linear(t1t, t2t, t1cet, flairt, self.weights).squeeze(0)  # (1,D,H,W)
        label = torch.from_numpy(segp).long()
        return {"image": fused, "label": label}  # Remove extra unsqueeze(0) to fix 6D tensor issue



In [ ]:
# 2-fold cross-validation (Kaggle T4 optimized for ~8h total)
data_root = "/kaggle/input/brats2015/BRATS2015/training"  # Update path for your Kaggle dataset

try:
    validate_braTS_root(data_root)
    cases = find_cases(data_root)
    folds = build_kfold_splits(cases, k=5, seed=42)

    # Per-modality Dice from paper Table IV (Yin et al., 2025)
    weights = compute_fusion_weights_from_val_dice(dice_t1=0.80, dice_t2=0.82, dice_t1ce=0.84, dice_flair=0.85)
    sampler = TumorPatchSampler(patch_size=(96,96,96), tumor_fraction=0.7)
    cfg = TrainConfig(epochs=60, batch_size=1, num_workers=0)

    for fold_id, fold in enumerate(folds[:2]):  # Run first 2 folds for CV under 8h
        print(f"\n===== Training fold {fold_id+1}/{2} =====")
        train_ds = BraTSDataset(fold["train"], weights, sampler, num_patches_per_volume=10, training=True)
        val_ds = BraTSDataset(fold["val"], weights, sampler, num_patches_per_volume=1, training=False)
        out_dir = os.path.join("./checkpoints/kfold", f"fold{fold_id}")
        run_training(train_ds, val_ds, num_classes=4, in_channels=1, out_dir=out_dir, cfg=cfg)
except Exception as e:
    print("Please set data_root to your BraTS dataset path and ensure required packages are installed.")
    print("Error:", e)
